In [ ]:
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

# 1. Configuración de rutas y carga de módulos base del proyecto
os.environ.setdefault("MTAH_DATA_DIR", r"D:\Proyectos\Barcelona project local\data")

_ROOT = Path.cwd()
if not (_ROOT / "games.py").exists() and (_ROOT.parent / "games.py").exists():
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import games
from src import data as D

PITCH_L, PITCH_W = games.GAMES[games.GAME]["pitch"]  # (105.3, 68.0) m


@dataclass
class ATDPreprocessConfig:
    fps: float = 25.0
    pitch_l: float = PITCH_L
    pitch_w: float = PITCH_W

    # Paso 3: Limpieza por slot (enmascaramiento)
    oob_margin_m: float = 1.5           # Margen permitido fuera de las líneas del campo (m)
    max_speed_mps: float = 12.0         # Umbral de salto físicamente imposible (12 m/s = 0.48 m/frame @ 25fps)
    min_streak_frames: int = 8          # Rachas visibles más cortas que esto se descartan como parpadeo

    # Paso 4: Interpolación de huecos cortos
    max_interp_gap_frames: int = 20     # Máximo hueco a interpolar (20 frames = 0.8s @ 25fps; >20 queda en NaN)

    # Paso 5: Suavizado Savitzky-Golay
    savgol_window: int = 7              # Ventana impar (~5-9 frames)
    savgol_polyorder: int = 2           # Orden polinomial

    # Paso 6: Calidad por frame
    min_reliable_players: int = 8       # Mínimo de jugadores visibles por equipo para alta confianza

    # Paso 7: Features colectivos
    stationary_speed_mps: float = 0.65  # Umbral para considerar a un jugador casi parado (m/s)


class ATDPreprocessor:
    """
    Pipeline de preprocesamiento puro del ATD (sin balón ni Viterbi):
      1. Rejilla completa 1..N_FRAMES y filtrado exclusivo a first_half y second_half.
      2. Conversión a metros y normalización de orientación (ambas mitades atacan igual).
      3. Enmascaramiento a NaN de fuera de campo, saltos >12 m/s (incluye swaps) y parpadeos.
      4. Interpolación lineal exclusiva de huecos cortos (<= max_interp_gap_frames).
      5. Suavizado Savitzky-Golay + cálculo propio de (vx, vy, speed, ax, ay, accel).
      6. Métricas de calidad por frame (n_visible_L, n_visible_A, low_conf_flag).
      7. Features colectivos independientes del balón por equipo (L y A).
    """

    def __init__(self, cfg: ATDPreprocessConfig):
        self.cfg = cfg

    @staticmethod
    def _get_half_masks(n_frames: int) -> Tuple[np.ndarray, np.ndarray]:
        """Obtiene máscaras booleanas de first_half y second_half desde el repositorio."""
        if hasattr(D, "half_masks"):
            return D.half_masks(n_frames)
        # Fallback usando intervalos estándar del partido si la función auxiliar cambia de nombre
        first_half = np.zeros(n_frames, dtype=bool)
        second_half = np.zeros(n_frames, dtype=bool)
        first_half[11185:90652] = True
        second_half[110808:190198] = True
        return first_half, second_half

    def _detect_first_half_home_attack_dir(
        self,
        xs_m: np.ndarray,
        slot_team: np.ndarray,
        first_half_mask: np.ndarray,
    ) -> int:
        """
        Determina si el equipo Local ('L', slot_team == 0) defiende x=0 y ataca hacia x=PITCH_L (+1)
        o defiende x=PITCH_L y ataca hacia x=0 (-1) en la primera parte.
        """
        home_slots = np.where(slot_team == 0)[0]
        away_slots = np.where(slot_team == 1)[0]
        med_x_L = np.nanmedian(xs_m[first_half_mask][:, home_slots])
        med_x_A = np.nanmedian(xs_m[first_half_mask][:, away_slots])
        return 1 if med_x_L <= med_x_A else -1

    @staticmethod
    def _mask_short_streaks(valid_mask: np.ndarray, min_len: int) -> np.ndarray:
        """Devuelve máscara booleana eliminando rachas True de longitud < min_len."""
        n = len(valid_mask)
        out = valid_mask.copy()
        i = 0
        while i < n:
            if out[i]:
                j = i
                while j < n and out[j]:
                    j += 1
                if (j - i) < min_len:
                    out[i:j] = False
                i = j
            else:
                i += 1
        return out

    def _clean_interpolate_and_smooth_slot(
        self,
        x_raw: np.ndarray,
        y_raw: np.ndarray,
        is_playing: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """Ejecuta los pasos 3, 4 y 5 sobre la serie temporal de un único slot."""
        n = len(x_raw)
        dt = 1.0 / self.cfg.fps
        max_step_m = self.cfg.max_speed_mps * dt

        x = x_raw.copy()
        y = y_raw.copy()

        # Enmascarar fuera de tiempo de juego
        x[~is_playing] = np.nan
        y[~is_playing] = np.nan

        # Paso 3a: Enmascarar fuera del campo (con margen pequeño)
        m = self.cfg.oob_margin_m
        oob = (x < -m) | (x > self.cfg.pitch_l + m) | (y < -m) | (y > self.cfg.pitch_w + m)
        x[oob] = np.nan
        y[oob] = np.nan

        # Paso 3b: Enmascarar saltos físicamente imposibles (> max_speed_mps, cubre ID swaps)
        # Se compara cada frame válido con el último frame válido previo para detectar saltos tras huecos
        valid = np.isfinite(x) & np.isfinite(y)
        last_v_idx = -1
        for t in range(n):
            if not valid[t]:
                continue
            if last_v_idx >= 0:
                gap_frames = t - last_v_idx
                dist = np.hypot(x[t] - x[last_v_idx], y[t] - y[last_v_idx])
                if dist > (max_step_m * gap_frames):
                    # El punto actual teletransportó -> marcar como NaN y no actualizar referencia
                    x[t] = np.nan
                    y[t] = np.nan
                    valid[t] = False
                    continue
            last_v_idx = t

        # Paso 3c: Descartar rachas visibles muy cortas (parpadeos)
        valid_clean = self._mask_short_streaks(valid, self.cfg.min_streak_frames)
        x[~valid_clean] = np.nan
        y[~valid_clean] = np.nan

        # Paso 4: Interpolación lineal solo de huecos cortos (<= max_interp_gap_frames)
        s_x = pd.Series(x)
        s_y = pd.Series(y)
        x_interp = s_x.interpolate(method="linear", limit=self.cfg.max_interp_gap_frames, limit_area="inside").to_numpy()
        y_interp = s_y.interpolate(method="linear", limit=self.cfg.max_interp_gap_frames, limit_area="inside").to_numpy()

        # Asegurar que no se interpoló cruzando el descanso (entre 1st_half y 2nd_half)
        x_interp[~is_playing] = np.nan
        y_interp[~is_playing] = np.nan

        # Paso 5: Suavizado Savitzky-Golay por tramo continuo y derivadas propias (v, a)
        x_smooth = np.full(n, np.nan, dtype=np.float64)
        y_smooth = np.full(n, np.nan, dtype=np.float64)
        vx = np.full(n, np.nan, dtype=np.float64)
        vy = np.full(n, np.nan, dtype=np.float64)
        ax = np.full(n, np.nan, dtype=np.float64)
        ay = np.full(n, np.nan, dtype=np.float64)

        win = self.cfg.savgol_window
        poly = self.cfg.savgol_polyorder

        is_fin = np.isfinite(x_interp) & np.isfinite(y_interp)
        i = 0
        while i < n:
            if is_fin[i]:
                j = i
                while j < n and is_fin[j]:
                    j += 1
                seg_len = j - i
                seg_x = x_interp[i:j]
                seg_y = y_interp[i:j]

                if seg_len >= win:
                    sx = savgol_filter(seg_x, window_length=win, polyorder=poly)
                    sy = savgol_filter(seg_y, window_length=win, polyorder=poly)
                elif seg_len >= 3:
                    w_odd = seg_len if seg_len % 2 == 1 else seg_len - 1
                    sx = savgol_filter(seg_x, window_length=w_odd, polyorder=min(poly, w_odd - 1))
                    sy = savgol_filter(seg_y, window_length=w_odd, polyorder=min(poly, w_odd - 1))
                else:
                    sx, sy = seg_x, seg_y

                x_smooth[i:j] = sx
                y_smooth[i:j] = sy

                if seg_len >= 2:
                    vx_seg = np.gradient(sx, dt)
                    vy_seg = np.gradient(sy, dt)
                    vx[i:j] = vx_seg
                    vy[i:j] = vy_seg
                    if seg_len >= 3:
                        ax[i:j] = np.gradient(vx_seg, dt)
                        ay[i:j] = np.gradient(vy_seg, dt)
                i = j
            else:
                i += 1

        speed = np.hypot(vx, vy)
        accel = np.hypot(ax, ay)
        return x_smooth, y_smooth, vx, vy, speed, accel, valid_clean

    def run(self, raw_atd_df: pd.DataFrame, slot_team: np.ndarray) -> Tuple[pd.DataFrame, dict]:
        t0 = time.perf_counter()
        n_total = D.N_FRAMES
        n_slots = len(slot_team)

        # ----------------------------------------------------------------------
        # PASO 1: Rejilla completa de frame_id (1..199319) y filtro de mitades
        # ----------------------------------------------------------------------
        full_grid = pd.DataFrame({"frame_id": np.arange(1, n_total + 1, dtype=np.int64)})
        df = full_grid.merge(raw_atd_df, on="frame_id", how="left")
        df["video_frame"] = df["frame_id"] - 1

        first_half, second_half = self._get_half_masks(n_total)
        df["period"] = np.where(first_half, "first_half", np.where(second_half, "second_half", "non_playing"))
        is_playing = first_half | second_half

        # ----------------------------------------------------------------------
        # PASO 2: Conversión a metros y orientación unificada entre mitades
        # ----------------------------------------------------------------------
        xs_m = np.full((n_total, n_slots), np.nan, dtype=np.float64)
        ys_m = np.full((n_total, n_slots), np.nan, dtype=np.float64)

        for s in range(n_slots):
            s_num = s + 1
            xs_m[:, s] = df[f"{s_num}_x"].to_numpy(dtype=np.float64) * self.cfg.pitch_l
            ys_m[:, s] = df[f"{s_num}_y"].to_numpy(dtype=np.float64) * self.cfg.pitch_w

        # Voltear ejes (x -> L - x, y -> W - y) en second_half para que cada equipo ataque siempre igual
        xs_m[second_half] = self.cfg.pitch_l - xs_m[second_half]
        ys_m[second_half] = self.cfg.pitch_w - ys_m[second_half]

        # Estandarizar adicionalmente para que Local ('L') siempre ataque de izq (0) a der (PITCH_L)
        home_attack_dir = self._detect_first_half_home_attack_dir(xs_m, slot_team, first_half)
        if home_attack_dir == -1:
            xs_m[is_playing] = self.cfg.pitch_l - xs_m[is_playing]
            ys_m[is_playing] = self.cfg.pitch_w - ys_m[is_playing]

        # ----------------------------------------------------------------------
        # PASOS 3, 4 y 5: Limpieza por slot, interpolación corta y Savitzky-Golay
        # ----------------------------------------------------------------------
        clean_x = np.full((n_total, n_slots), np.nan, dtype=np.float64)
        clean_y = np.full((n_total, n_slots), np.nan, dtype=np.float64)
        clean_vx = np.full((n_total, n_slots), np.nan, dtype=np.float64)
        clean_vy = np.full((n_total, n_slots), np.nan, dtype=np.float64)
        clean_spd = np.full((n_total, n_slots), np.nan, dtype=np.float64)
        clean_acc = np.full((n_total, n_slots), np.nan, dtype=np.float64)
        raw_valid_mask = np.zeros((n_total, n_slots), dtype=bool)

        for s in range(n_slots):
            cx, cy, cvx, cvy, cspd, cacc, vmask = self._clean_interpolate_and_smooth_slot(
                xs_m[:, s], ys_m[:, s], is_playing
            )
            clean_x[:, s] = cx
            clean_y[:, s] = cy
            clean_vx[:, s] = cvx
            clean_vy[:, s] = cvy
            clean_spd[:, s] = cspd
            clean_acc[:, s] = cacc
            raw_valid_mask[:, s] = vmask

        # Filtrar tabla exclusivamente a tiempo de juego (first_half y second_half)
        play_idx = np.where(is_playing)[0]
        out_df = pd.DataFrame({
            "frame_id": df.loc[play_idx, "frame_id"].to_numpy(),
            "video_frame": df.loc[play_idx, "video_frame"].to_numpy(),
            "time_s": np.round(df.loc[play_idx, "video_frame"].to_numpy() / self.cfg.fps, 4),
            "period": df.loc[play_idx, "period"].to_numpy(),
        })

        # Guardar columnas limpias por slot (con sufijo de equipo L/A en metadato)
        slot_cols_dict = {}
        for s in range(n_slots):
            s_num = s + 1
            slot_cols_dict[f"{s_num}_x_m"] = np.round(clean_x[play_idx, s], 3)
            slot_cols_dict[f"{s_num}_y_m"] = np.round(clean_y[play_idx, s], 3)
            slot_cols_dict[f"{s_num}_vx"] = np.round(clean_vx[play_idx, s], 3)
            slot_cols_dict[f"{s_num}_vy"] = np.round(clean_vy[play_idx, s], 3)
            slot_cols_dict[f"{s_num}_speed"] = np.round(clean_spd[play_idx, s], 3)
            slot_cols_dict[f"{s_num}_accel"] = np.round(clean_acc[play_idx, s], 3)
        out_df = pd.concat([out_df, pd.DataFrame(slot_cols_dict)], axis=1)

        # ----------------------------------------------------------------------
        # PASO 6: Calidad por frame (Local 'L' = slot_team 0, Away 'A' = slot_team 1)
        # ----------------------------------------------------------------------
        idx_L = np.where(slot_team == 0)[0]
        idx_A = np.where(slot_team == 1)[0]

        x_play = clean_x[play_idx]
        y_play = clean_y[play_idx]
        spd_play = clean_spd[play_idx]

        # Conteo de jugadores disponibles (tras limpieza e interpolación corta)
        n_avail_L = np.isfinite(x_play[:, idx_L]).sum(axis=1)
        n_avail_A = np.isfinite(x_play[:, idx_A]).sum(axis=1)
        # Conteo de lecturas directas sin interpolar
        n_obs_L = raw_valid_mask[play_idx][:, idx_L].sum(axis=1)
        n_obs_A = raw_valid_mask[play_idx][:, idx_A].sum(axis=1)

        low_conf_L = n_avail_L < self.cfg.min_reliable_players
        low_conf_A = n_avail_A < self.cfg.min_reliable_players
        low_conf_any = low_conf_L | low_conf_A

        # ----------------------------------------------------------------------
        # PASO 7: Features colectivos independientes del balón (L y A)
        # ----------------------------------------------------------------------
        def _team_collectives(x_mat: np.ndarray, y_mat: np.ndarray, s_mat: np.ndarray, attack_dir: int):
            cx = np.nanmean(x_mat, axis=1)
            cy = np.nanmean(y_mat, axis=1)
            # Dispersión radial (RMS distance al centroide del equipo)
            spread = np.sqrt(np.nanmean((x_mat - cx[:, None]) ** 2 + (y_mat - cy[:, None]) ** 2, axis=1))
            mean_spd = np.nanmean(s_mat, axis=1)
            valid_cnt = np.maximum(1, np.isfinite(s_mat).sum(axis=1))
            frac_stat = (s_mat < self.cfg.stationary_speed_mps).sum(axis=1) / valid_cnt

            # Altura de la línea defensiva (percentil 15 en dirección de ataque, excluyendo al portero más retrasado)
            x_sorted = np.sort(x_mat, axis=1) if attack_dir == 1 else np.sort(-x_mat, axis=1)
            # Índice 1 corresponde al segundo jugador más retrasado (último defensa tras el GK)
            def_line = x_sorted[:, 1] if attack_dir == 1 else -x_sorted[:, 1]
            return cx, cy, spread, mean_spd, frac_stat, def_line

        cx_L, cy_L, spread_L, spd_L, stat_L, def_L = _team_collectives(
            x_play[:, idx_L], y_play[:, idx_L], spd_play[:, idx_L], attack_dir=1
        )
        cx_A, cy_A, spread_A, spd_A, stat_A, def_A = _team_collectives(
            x_play[:, idx_A], y_play[:, idx_A], spd_play[:, idx_A], attack_dir=-1
        )

        # Distancia entre centroides (eje X y euclidiana) + compresión de bloque
        centroid_dist_x = cx_A - cx_L
        centroid_dist_euclid = np.hypot(cx_A - cx_L, cy_A - cy_L)

        team_features_df = pd.DataFrame({
            # Paso 6: Calidad
            "n_obs_L": n_obs_L,
            "n_obs_A": n_obs_A,
            "n_visible_L": n_avail_L,
            "n_visible_A": n_avail_A,
            "low_conf_L": low_conf_L,
            "low_conf_A": low_conf_A,
            "low_conf_frame": low_conf_any,
            # Paso 7: Colectivos L y A
            "centroid_x_L": np.round(cx_L, 3),
            "centroid_y_L": np.round(cy_L, 3),
            "spread_L": np.round(spread_L, 3),
            "mean_speed_L": np.round(spd_L, 3),
            "frac_stationary_L": np.round(stat_L, 4),
            "def_line_x_L": np.round(def_L, 3),
            "centroid_x_A": np.round(cx_A, 3),
            "centroid_y_A": np.round(cy_A, 3),
            "spread_A": np.round(spread_A, 3),
            "mean_speed_A": np.round(spd_A, 3),
            "frac_stationary_A": np.round(stat_A, 4),
            "def_line_x_A": np.round(def_A, 3),
            "centroid_dist_x": np.round(centroid_dist_x, 3),
            "centroid_dist_euclid": np.round(centroid_dist_euclid, 3),
        })

        out_df = pd.concat([out_df, team_features_df], axis=1)

        summary = {
            "elapsed_s": round(time.perf_counter() - t0, 2),
            "total_match_frames": n_total,
            "playing_frames": len(out_df),
            "first_half_frames": int(first_half.sum()),
            "second_half_frames": int(second_half.sum()),
            "home_first_half_dir": home_attack_dir,
            "avg_visible_L": round(float(n_avail_L.mean()), 2),
            "avg_visible_A": round(float(n_avail_A.mean()), 2),
            "low_conf_pct": round(float(100.0 * low_conf_any.mean()), 2),
        }
        return out_df, summary


# ==============================================================================
# EJECUCIÓN DE CELDA 1: Cargar ATD crudo -> Preprocesar (Pasos 1..7)
# ==============================================================================
raw_atd_df = D.load_tracking()
slot_team = D.load_slot_teams()  # 0 = Local ('L'), 1 = Away ('A')

atd_cfg = ATDPreprocessConfig(
    oob_margin_m=1.5,
    max_speed_mps=12.0,
    min_streak_frames=8,
    max_interp_gap_frames=20,
    savgol_window=7,
    min_reliable_players=8,
)

preprocessor = ATDPreprocessor(atd_cfg)
df_atd_clean, atd_summary = preprocessor.run(raw_atd_df, slot_team)

print("=== RESUMEN PREPROCESAMIENTO ATD (PASOS 1 A 7) ===")
for k, v in atd_summary.items():
    print(f"  {k:22s}: {v}")

print("\nMuestra de features colectivos y calidad por frame:")
cols_preview = [
    "frame_id", "video_frame", "period",
    "n_visible_L", "n_visible_A", "low_conf_frame",
    "centroid_x_L", "centroid_x_A", "def_line_x_L", "def_line_x_A",
    "mean_speed_L", "frac_stationary_L",
]
print(df_atd_clean[cols_preview].head(8).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# ==============================================================================
# 1. VERIFICACIÓN DE ORIENTACIÓN ENTRE MITADES Y CALIDAD DE DATOS (PASOS 2, 6, 7)
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
plt.subplots_adjust(hspace=0.32, wspace=0.22)

fh = df_atd_clean[df_atd_clean["period"] == "first_half"]
sh = df_atd_clean[df_atd_clean["period"] == "second_half"]

# Panel A: Centroides X en 1ª vs 2ª parte (ambas deben mostrar L atacando hacia 105.3m)
ax = axes[0, 0]
ax.hist(fh["centroid_x_L"].dropna(), bins=50, alpha=0.55, color="#1f77b4", label="L - 1ª Parte", density=True)
ax.hist(sh["centroid_x_L"].dropna(), bins=50, alpha=0.55, color="#00b4d8", label="L - 2ª Parte", density=True)
ax.hist(fh["centroid_x_A"].dropna(), bins=50, alpha=0.55, color="#d62728", label="A - 1ª Parte", density=True)
ax.hist(sh["centroid_x_A"].dropna(), bins=50, alpha=0.55, color="#ff7f0e", label="A - 2ª Parte", density=True)
ax.axvline(PITCH_L / 2, color="k", linestyle="--", linewidth=1, alpha=0.6)
ax.set_title("Verificación Paso 2: Centroides X por Mitad (L -> 105.3m | A -> 0m)")
ax.set_xlabel("Posición X en el campo (m)")
ax.set_ylabel("Densidad")
ax.legend(fontsize=8, ncol=2)

# Panel B: Jugadores disponibles por equipo a lo largo del partido (Paso 6)
ax = axes[0, 1]
step_plot = 25  # Submuestreo a 1s para visualización limpia
ax.plot(df_atd_clean["video_frame"].iloc[::step_plot], df_atd_clean["n_visible_L"].iloc[::step_plot],
        color="#1f77b4", alpha=0.75, linewidth=0.9, label="Disponibles Local (L)")
ax.plot(df_atd_clean["video_frame"].iloc[::step_plot], df_atd_clean["n_visible_A"].iloc[::step_plot],
        color="#d62728", alpha=0.75, linewidth=0.9, label="Disponibles Away (A)")
ax.axhline(atd_cfg.min_reliable_players, color="orange", linestyle="--", linewidth=1.2,
           label=f"Umbral baja confianza (<{atd_cfg.min_reliable_players})")
ax.set_title("Verificación Paso 6: Cobertura de Jugadores por Equipo")
ax.set_xlabel("video_frame")
ax.set_ylabel("Nº Jugadores")
ax.set_ylim(0, 13)
ax.legend(fontsize=8)

# Panel C: Distribución de velocidad propia vs velocidad cruda del ATD (Paso 5)
ax = axes[1, 0]
spd_cols = [f"{s}_speed" for s in range(1, 47)]
clean_speeds = df_atd_clean[spd_cols].to_numpy().ravel()
clean_speeds = clean_speeds[np.isfinite(clean_speeds)]

raw_spd_cols = [f"{s}_s" for s in range(1, 47)]
raw_speeds = raw_atd_df[raw_spd_cols].to_numpy().ravel()
raw_speeds = raw_speeds[np.isfinite(raw_speeds) & (raw_speeds <= 20.0)]

ax.hist(raw_speeds, bins=60, range=(0, 13), alpha=0.45, color="gray", density=True, label="Velocidad cruda ATD ({i}_s)")
ax.hist(clean_speeds, bins=60, range=(0, 13), alpha=0.65, color="#2ca02c", density=True, label="Velocidad Savitzky-Golay propia")
ax.axvline(atd_cfg.max_speed_mps, color="red", linestyle="--", linewidth=1, label=f"Corte físico ({atd_cfg.max_speed_mps} m/s)")
ax.set_title("Verificación Paso 5: Velocidad Suavizada vs Cruda")
ax.set_xlabel("Velocidad (m/s)")
ax.set_ylabel("Densidad")
ax.legend(fontsize=8)

# Panel D: Relación entre velocidad media colectiva y jugadores estacionarios (Paso 7 - señal de Dead Ball 'D')
ax = axes[1, 1]
sample_idx = np.linspace(0, len(df_atd_clean) - 1, 3000, dtype=int)
sub_vis = df_atd_clean.iloc[sample_idx]
mean_spd_both = 0.5 * (sub_vis["mean_speed_L"] + sub_vis["mean_speed_A"])
frac_stat_both = 0.5 * (sub_vis["frac_stationary_L"] + sub_vis["frac_stationary_A"])
sc = ax.scatter(mean_spd_both, frac_stat_both, c=sub_vis["centroid_dist_euclid"], cmap="viridis", s=8, alpha=0.6)
plt.colorbar(sc, ax=ax, label="Dist. entre centroides (m)")
ax.set_title("Verificación Paso 7: Dinámica Colectiva (Detección de Juego Parado 'D')")
ax.set_xlabel("Velocidad media colectiva (m/s)")
ax.set_ylabel("Fracción de jugadores casi parados (<0.65 m/s)")

plt.show()

# ==============================================================================
# 2. INSPECCIÓN DE SLOTS ACTIVOS POR EQUIPO Y EXPORTACIÓN
# ==============================================================================
slot_stats = []
for s in range(1, len(slot_team) + 1):
    vis_frames = int(df_atd_clean[f"{s}_x_m"].notna().sum())
    if vis_frames > 0:
        slot_stats.append({
            "slot": s,
            "team": "L" if slot_team[s - 1] == 0 else "A",
            "valid_frames": vis_frames,
            "coverage_pct": round(100.0 * vis_frames / len(df_atd_clean), 2),
            "mean_speed_mps": round(float(df_atd_clean[f"{s}_speed"].mean()), 2),
            "max_speed_mps": round(float(df_atd_clean[f"{s}_speed"].max()), 2),
        })

df_slot_summary = pd.DataFrame(slot_stats).sort_values(["team", "valid_frames"], ascending=[True, False])
print("=== COBERTURA Y CINEMÁTICA POR SLOT TRAS LIMPIEZA ===")
print(df_slot_summary.to_string(index=False))

# Exportar tabla limpia (Parquet es ~10x más rápido y ligero que CSV para 158k filas x 295 cols)
OUT_DIR = Path(os.environ["MTAH_DATA_DIR"]).parent / "processed_atd"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = OUT_DIR / f"{games.GAME}_atd_clean_LA.parquet"
META_PATH = OUT_DIR / f"{games.GAME}_slot_teams_LA.csv"

df_atd_clean.to_parquet(PARQUET_PATH, index=False)
df_slot_summary.to_csv(META_PATH, index=False)

print(f"\nATD preprocesado guardado en : {PARQUET_PATH} ({PARQUET_PATH.stat().st_size / (1024**2):.1f} MB)")
print(f"Resumen de slots guardado en : {META_PATH}")

In [ ]:
@dataclass
class BallFeaturesConfig:
    fps: float = 25.0
    radii_m: Tuple[float, float, float] = (3.0, 5.0, 10.0)
    min_dist_eps: float = 1e-4  # Evita división por cero al normalizar el vector unitario hacia el balón


def compute_ball_relative_features(
    df_atd: pd.DataFrame,
    slot_team: np.ndarray,
    ball_df: pd.DataFrame,
    ball_x_col: str = "ball_x_m",
    ball_y_col: str = "ball_y_m",
    ball_vx_col: Optional[str] = "ball_vx",
    ball_vy_col: Optional[str] = "ball_vy",
    cfg: BallFeaturesConfig = BallFeaturesConfig(),
) -> pd.DataFrame:
    """
    Paso 8: Sincroniza el balón preprocesado (ya en metros y con la misma orientación de mitades
    que df_atd) por 'video_frame' y calcula los features espaciales y cinemáticos respecto al balón.
    Cuando el balón está ausente, deja las métricas en NaN y marca ball_available = False.
    """
    # 1. Sincronización estricta por video_frame sobre la rejilla de tiempo de juego del ATD
    merge_cols = ["video_frame", ball_x_col, ball_y_col]
    has_ball_vel = (
        ball_vx_col is not None
        and ball_vy_col is not None
        and ball_vx_col in ball_df.columns
        and ball_vy_col in ball_df.columns
    )
    if has_ball_vel:
        merge_cols += [ball_vx_col, ball_vy_col]

    ball_sync = (
        df_atd[["video_frame"]]
        .merge(ball_df[merge_cols].drop_duplicates("video_frame"), on="video_frame", how="left")
    )

    bx = ball_sync[ball_x_col].to_numpy(dtype=np.float64)
    by = ball_sync[ball_y_col].to_numpy(dtype=np.float64)
    ball_available = np.isfinite(bx) & np.isfinite(by)

    if has_ball_vel:
        bvx = np.where(ball_available, ball_sync[ball_vx_col].to_numpy(dtype=np.float64), np.nan)
        bvy = np.where(ball_available, ball_sync[ball_vy_col].to_numpy(dtype=np.float64), np.nan)
    else:
        # Derivada central sobre tramos disponibles si el dataframe del balón no trae (vx, vy)
        bvx = np.full_like(bx, np.nan)
        bvy = np.full_like(by, np.nan)
        dt = 1.0 / cfg.fps
        i, n_b = 0, len(bx)
        while i < n_b:
            if ball_available[i]:
                j = i
                while j < n_b and ball_available[j]:
                    j += 1
                if (j - i) >= 2:
                    bvx[i:j] = np.gradient(bx[i:j], dt)
                    bvy[i:j] = np.gradient(by[i:j], dt)
                else:
                    bvx[i:j] = 0.0
                    bvy[i:j] = 0.0
                i = j
            else:
                i += 1

    bvx = np.nan_to_num(bvx, nan=0.0)
    bvy = np.nan_to_num(bvy, nan=0.0)

    n_frames = len(df_atd)
    n_slots = len(slot_team)

    # Extraer matrices (T, 46) de posiciones y velocidades limpias del ATD
    px = np.column_stack([df_atd[f"{s}_x_m"].to_numpy(dtype=np.float64) for s in range(1, n_slots + 1)])
    py = np.column_stack([df_atd[f"{s}_y_m"].to_numpy(dtype=np.float64) for s in range(1, n_slots + 1)])
    pvx = np.column_stack([df_atd[f"{s}_vx"].to_numpy(dtype=np.float64) for s in range(1, n_slots + 1)])
    pvy = np.column_stack([df_atd[f"{s}_vy"].to_numpy(dtype=np.float64) for s in range(1, n_slots + 1)])

    # Vector desde cada jugador hacia el balón: r = p_ball - p_player
    rx = bx[:, None] - px
    ry = by[:, None] - py
    dist = np.hypot(rx, ry)  # (T, 46); será NaN donde falte el balón o el jugador

    # Velocidad de acercamiento (closing speed > 0 significa que la distancia se reduce):
    # v_closing = ((v_player - v_ball) · r) / ||r||
    safe_dist = np.maximum(dist, cfg.min_dist_eps)
    ux = rx / safe_dist
    uy = ry / safe_dist
    closing_spd_all = (pvx - bvx[:, None]) * ux + (pvy - bvy[:, None]) * uy

    idx_L = np.where(slot_team == 0)[0]
    idx_A = np.where(slot_team == 1)[0]

    def _team_ball_metrics(team_idx: np.ndarray):
        d_tm = dist[:, team_idx]
        c_tm = closing_spd_all[:, team_idx]
        has_any_player = np.isfinite(d_tm).any(axis=1) & ball_available

        d_min = np.full(n_frames, np.nan, dtype=np.float64)
        closing_nearest = np.full(n_frames, np.nan, dtype=np.float64)
        nearest_slot = np.full(n_frames, -1, dtype=np.int32)

        valid_rows = np.where(has_any_player)[0]
        if len(valid_rows) > 0:
            d_sub = np.where(np.isfinite(d_tm[valid_rows]), d_tm[valid_rows], np.inf)
            local_arg = np.argmin(d_sub, axis=1)
            d_min[valid_rows] = d_sub[np.arange(len(valid_rows)), local_arg]
            closing_nearest[valid_rows] = c_tm[valid_rows, local_arg]
            nearest_slot[valid_rows] = team_idx[local_arg] + 1  # Slot 1-based

        # Conteos en radios (3m, 5m, 10m): NaN cuando ball_available es False
        counts = {}
        for r in cfg.radii_m:
            cnt = np.where(ball_available, (d_tm <= r).sum(axis=1).astype(np.float64), np.nan)
            counts[r] = cnt

        return d_min, closing_nearest, nearest_slot, counts

    d_min_L, close_L, slot_near_L, counts_L = _team_ball_metrics(idx_L)
    d_min_A, close_A, slot_near_A, counts_A = _team_ball_metrics(idx_A)

    # Diferencia local entre equipos (d_home - d_away): negativa -> Local más cerca; positiva -> Away más cerca
    diff_d_min_LA = d_min_L - d_min_A

    # Flag explícito para degradar la emisión del Viterbi a los features colectivos (Paso 7) y transición
    low_conf = df_atd["low_conf_frame"].to_numpy(dtype=bool) if "low_conf_frame" in df_atd.columns else False
    use_team_fallback = (~ball_available) | low_conf

    ball_feats = pd.DataFrame({
        "ball_x_m": np.round(bx, 3),
        "ball_y_m": np.round(by, 3),
        "ball_available": ball_available,
        "use_team_fallback": use_team_fallback,
        # Distancias mínimas y diferencia local (d_home - d_away)
        "d_min_L": np.round(d_min_L, 3),
        "d_min_A": np.round(d_min_A, 3),
        "diff_d_min_LA": np.round(diff_d_min_LA, 3),
        "nearest_slot_L": slot_near_L,
        "nearest_slot_A": slot_near_A,
        # Velocidad de acercamiento de los más cercanos (+ m/s = cerrando espacio hacia el balón)
        "closing_speed_L": np.round(close_L, 3),
        "closing_speed_A": np.round(close_A, 3),
        # Conteos de presión en radios de 3, 5 y 10 m
        "count_3m_L": counts_L[3.0],
        "count_5m_L": counts_L[5.0],
        "count_10m_L": counts_L[10.0],
        "count_3m_A": counts_A[3.0],
        "count_5m_A": counts_A[5.0],
        "count_10m_A": counts_A[10.0],
    })

    # Evitar duplicados si re-ejecutas la celda sobre el mismo DataFrame
    base_df = df_atd.drop(columns=[c for c in ball_feats.columns if c in df_atd.columns])
    return pd.concat([base_df.reset_index(drop=True), ball_feats], axis=1)

In [ ]:
df_ball_clean = pd.read_csv(os.environ["MTAH_DATA_DIR"] + r"\processed\ball_tracking_csv\ball_tracking_all.csv")

In [ ]:
df_ball_clean.head()

In [ ]:
# 1. Comparar cada fila completa contra los nombres de las columnas y eliminar las que coincidan
is_header_row = df_ball_clean.astype(str).apply(lambda col: col.str.strip() == col.name).all(axis=1)
df = df_ball_clean[~is_header_row].reset_index(drop=True)

# 2. Restaurar los tipos numéricos (porque los headers repetidos forzaron todo a string/object)
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except (ValueError, TypeError):
        pass

In [ ]:
df_ball_clean.head(10)

In [ ]:
df_ball_clean.columns

In [ ]:
df_ball_clean["frame"] = pd.to_numeric(df_ball_clean["frame"], errors="coerce")
df_ball_clean = (
    df_ball_clean.dropna(subset=["frame"])
    .assign(frame=lambda d: d["frame"].astype(int))
    .sort_values(by="frame", ascending=True)
    .reset_index(drop=True)
)

df_ball_clean.head(10)  # Primeros 10 frames (de menor a mayor)

In [ ]:
#Cambiar el nombre de frame a video_frame para que coincida con el df_atd_clean
df_ball_clean = df_ball_clean.rename(columns={"frame": "video_frame"})

In [ ]:
df_ball_clean.head(5)

In [ ]:
def prepare_and_sync_ball(
    df_atd_clean: pd.DataFrame,
    slot_team: np.ndarray,
    df_ball_clean: pd.DataFrame,
    atd_summary: dict,
    max_ball_gap_steps: int = 20,   # Descarta predicciones/huecos largos (>20 frames = 0.8s)
    oob_margin_m: float = 2.0,
) -> pd.DataFrame:
    ball = df_ball_clean.copy()

    # 1. Limpiar posibles headers repetidos de concatenación y asegurar tipos numéricos
    ball = ball[ball["video_frame"].astype(str).str.strip() != "video_frame"].copy()
    num_cols = [
        "video_frame", "time_s", "pred_x", "pred_y",
        "pitch_x_norm", "pitch_y_norm", "pitch_x_m", "pitch_y_m",
        "score", "gap_steps", "speed_px_s",
    ]
    for c in num_cols:
        if c in ball.columns:
            ball[c] = pd.to_numeric(ball[c], errors="coerce")

    ball = (
        ball.dropna(subset=["video_frame"])
        .assign(video_frame=lambda d: d["video_frame"].astype(np.int64))
        .sort_values("video_frame")
        .drop_duplicates("video_frame", keep="first")
        .reset_index(drop=True)
    )

    # 2. Reescalar usando las dimensiones exactas del campo del partido (PITCH_L x PITCH_W)
    if "pitch_x_norm" in ball.columns and "pitch_y_norm" in ball.columns:
        bx = ball["pitch_x_norm"].to_numpy(dtype=np.float64) * PITCH_L
        by = ball["pitch_y_norm"].to_numpy(dtype=np.float64) * PITCH_W
    else:
        bx = ball["pitch_x_m"].to_numpy(dtype=np.float64)
        by = ball["pitch_y_m"].to_numpy(dtype=np.float64)

    # 3. Enmascarar frames sin balón (state == 'NONE'), huecos largos (gap_steps > max) o fuera de campo
    state_str = ball["state"].astype(str).str.strip().str.upper()
    invalid_state = state_str.isin(["NONE", "MISSING", "LOST", "NAN", ""])
    long_gap = (ball["gap_steps"].fillna(999).to_numpy() > max_ball_gap_steps) if "gap_steps" in ball.columns else False
    oob = (bx < -oob_margin_m) | (bx > PITCH_L + oob_margin_m) | (by < -oob_margin_m) | (by > PITCH_W + oob_margin_m)

    mask_invalid = invalid_state.to_numpy() | long_gap | oob | ~np.isfinite(bx) | ~np.isfinite(by)
    bx[mask_invalid] = np.nan
    by[mask_invalid] = np.nan

    ball["ball_x_m"] = bx
    ball["ball_y_m"] = by

    # 4. Sincronizar con df_atd_clean y aplicar la misma orientación de mitades (Paso 2)
    merged_period = df_atd_clean[["video_frame", "period"]].merge(
        ball[["video_frame", "ball_x_m", "ball_y_m", "state", "score"]],
        on="video_frame",
        how="left",
    )

    is_second = (merged_period["period"] == "second_half").to_numpy()
    bx_aligned = merged_period["ball_x_m"].to_numpy(dtype=np.float64)
    by_aligned = merged_period["ball_y_m"].to_numpy(dtype=np.float64)

    # Voltear segunda parte para que coincida con df_atd_clean
    bx_aligned[is_second] = PITCH_L - bx_aligned[is_second]
    by_aligned[is_second] = PITCH_W - by_aligned[is_second]

    if atd_summary.get("home_first_half_dir", 1) == -1:
        bx_aligned = PITCH_L - bx_aligned
        by_aligned = PITCH_W - by_aligned

    ball_aligned_df = pd.DataFrame({
        "video_frame": merged_period["video_frame"],
        "ball_x_m": bx_aligned,
        "ball_y_m": by_aligned,
    })

    # 5. Calcular features relativos al balón (Paso 8)
    df_out = compute_ball_relative_features(
        df_atd=df_atd_clean,
        slot_team=slot_team,
        ball_df=ball_aligned_df,
        ball_x_col="ball_x_m",
        ball_y_col="ball_y_m",
        ball_vx_col=None,
        ball_vy_col=None,
    )
    df_out["ball_state"] = merged_period["state"].fillna("NONE").to_numpy()
    df_out["ball_score"] = merged_period["score"].to_numpy()
    return df_out


# ==============================================================================
# EJECUCIÓN
# ==============================================================================
df_full = prepare_and_sync_ball(
    df_atd_clean=df_atd_clean,
    slot_team=slot_team,
    df_ball_clean=df_ball_clean,
    atd_summary=atd_summary,
    max_ball_gap_steps=20,
)

print(f"Total frames de juego        : {len(df_full)}")
print(f"Balón disponible             : {df_full['ball_available'].mean() * 100:.2f}%")
print(f"Fallback a features equipo   : {df_full['use_team_fallback'].mean() * 100:.2f}%")

cols_check = [
    "video_frame", "period", "ball_state", "ball_available",
    "ball_x_m", "ball_y_m", "d_min_L", "d_min_A", "diff_d_min_LA",
    "closing_speed_L", "closing_speed_A", "count_3m_L", "count_3m_A",
]
print(df_full[df_full["ball_available"]][cols_check].head(8).to_string(index=False))

In [ ]:
df_full.head(5)

In [ ]:
# Verificación rápida: en frames con balón disponible, el jugador más cercano debe estar cerca (< 3-5 m en mediana)
print("Mediana distancia mínima L/A al balón en 1ª parte:",
      df_full[df_full["period"] == "first_half"][["d_min_L", "d_min_A"]].min(axis=1).median())
print("Mediana distancia mínima L/A al balón en 2ª parte:",
      df_full[df_full["period"] == "second_half"][["d_min_L", "d_min_A"]].min(axis=1).median())

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path(os.environ["MTAH_DATA_DIR"]) / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_OUT = PROCESSED_DIR / f"{games.GAME}_atd_ball_features_LA.parquet"
CSV_OUT = PROCESSED_DIR / f"{games.GAME}_atd_ball_features_LA.csv"
SLOTS_OUT = PROCESSED_DIR / f"{games.GAME}_slot_teams_LA.csv"

# 1. Exportar dataset completo (Pasos 1 a 8) en Parquet y CSV
df_full.to_parquet(PARQUET_OUT, index=False)
df_full.to_csv(CSV_OUT, index=False)

# 2. Exportar mapa de correspondencia slot -> equipo ('L' / 'A')
pd.DataFrame({
    "slot": np.arange(1, len(slot_team) + 1),
    "team": np.where(slot_team == 0, "L", "A"),
}).to_csv(SLOTS_OUT, index=False)

print("=== EXPORTACIÓN COMPLETADA ===")
print(f"  Parquet : {PARQUET_OUT} ({PARQUET_OUT.stat().st_size / (1024**2):.2f} MB)")
print(f"  CSV     : {CSV_OUT} ({CSV_OUT.stat().st_size / (1024**2):.2f} MB)")
print(f"  Slots   : {SLOTS_OUT}")
print(f"  Shape   : {df_full.shape[0]} filas x {df_full.shape[1]} columnas")